# Energy champions — feature importance (train-only, clean split)

Champions from `selection_table.csv` — purged inner k-fold + 1SE, global cut **2021-10-06**.
All importance analysis uses **train data only** (post-split). The held-out 30% is **sealed**.

| Instrument | Group | Champion | AUC | Lower CI | Signal |
|------------|-------|----------|-----|----------|--------|
| cl1s | cl1s | XGB | 0.675±0.139 | 0.54 | YES |
| ho1s | energy_all | Logistic | 0.800±0.303 | 0.50 | NO |
| rb1s | energy_all | Logistic | 0.551±0.094 | 0.46 | NO |
| ng1s | energy_all | RF | 0.477±0.259 | 0.22 | NO |

> **ho1s, rb1s, ng1s** carry no confirmed signal. ho1s/rb1s/ng1s run on the pooled `energy_all`
> group; importance is scored on each instrument's own CPCV test slice. ho1s shows very high
> AUC variance (±0.303) driven by a few folds — treat point estimates with caution.
> ng1s had only ~28 train events at the global cut so 3 of 15 CPCV folds were skipped.


In [ ]:
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display

BASE = Path('outputs/importance')

def show(path, width=1100):
    p = Path(path)
    if p.exists():
        display(Image(str(p), width=width))
    else:
        print(f'[missing] {p}')

def load(path):
    p = Path(path)
    if p.exists():
        return pd.read_csv(p)
    print(f'[missing] {p}')
    return pd.DataFrame()

def cluster_summary(inst):
    mem = load(BASE / inst / 'cluster_membership.csv')
    mda = load(BASE / inst / 'clustered_mda_full.csv')
    if mem.empty or mda.empty:
        return pd.DataFrame()
    grp = (
        mem.groupby('cluster')
        .agg(
            n_members=('feature', 'count'),
            dominant_pfx=('f_prefix', lambda x: x.value_counts().index[0]),
            purity=('f_prefix', lambda x: round(x.value_counts().iloc[0] / len(x), 2)),
        )
        .reset_index()
    )
    mda_sub = mda[['cluster', 'mean_drop', 'std_drop', 'significant']].copy()
    return grp.merge(mda_sub, on='cluster', how='left').sort_values('mean_drop', ascending=False)

print('Setup complete. Artifact root:', BASE.resolve())


---
## CL1S — champion: `cl1s` / XGB

**CPCV:** 15 paths · AUC 0.697 ± 0.139 · **SIGNAL** (lower CI 0.54)

**Significant clusters:** C14\_f11 only (MDA 0.224 ± 0.115) — dominates all others by ~50×.

> CL1S is dominated by a single cluster of cross-asset macro/volatility features. Within C14_f11,
> `f11_move_z` (MOVE bond volatility index z-score, SHAP 0.219) and `f2_vol_60` (60-day realised
> volatility, SHAP 0.195) are near-equally important, loading on opposite signs of PC1 —
> a single volatility-regime latent factor explains 67% of the cluster's variance.
> Notably, **F5_signal is slightly negative** (−0.004 MDA) for CL1S, meaning internal
> signal-quality features are uninformative or even detrimental — crude oil predictability
> comes entirely from external market stress signals, not signal metadata.
> MDI–SHAP agreement is strong (τ=0.71) but MDA disagrees with both (τ≈0.33/0.28),
> reflecting the sharp concentration: one cluster dominates MDA; MDI/SHAP spread across more.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('cl1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.05, vmax=0.25)
    .set_caption('cl1s — cluster summary (K=15 corr + 3 hand-assigned = 18 groups)')
)
show(BASE / 'cl1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

Clustered MDA (dark = significant > 1σ). Cross-check with MDI and SHAP.


In [ ]:
show(BASE / 'cl1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'cl1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'cl1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    for c in ['mdi_sum', 'shap_sum']:
        if c in cc.columns: fmt[c] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.25)
        .set_caption('cl1s — cluster cross-check (MDA · MDI · SHAP ranks)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by MDA)

Members ranked by mean |SHAP|. PCA on train submatrix.

**C14_f11** (the only significant cluster): `f11_move_z` and `f2_vol_60` dominate.
PC1 explains 67% — a single volatility-regime dimension.
`f11_move_z` and `f2_vol_60` load on opposite PC1 signs, suggesting they capture
complementary aspects of volatility (external fear index vs. realised crude vol).


In [ ]:
wc = load(BASE / 'cl1s' / 'within_cluster_C14_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'cl1s | C14_f11 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'cl1s' / 'within_cluster_C14_f11.png', width=950)


In [ ]:
wc = load(BASE / 'cl1s' / 'within_cluster_C15_f2.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'cl1s | C15_f2 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'cl1s' / 'within_cluster_C15_f2.png', width=950)


In [ ]:
wc = load(BASE / 'cl1s' / 'within_cluster_C9_f1.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'cl1s | C9_f1 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'cl1s' / 'within_cluster_C9_f1.png', width=950)


### Step 5 · Global per-feature SHAP

Mean |SHAP| per feature, averaged across 15 CPCV paths. Red = positive, blue = negative predictive direction.


In [ ]:
gs = load(BASE / 'cl1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('cl1s — top 25 features by mean|SHAP|')
    )
show(BASE / 'cl1s' / 'global_shap_chart.png', width=1000)


---
## HO1S — champion: `energy_all` / Logistic (elastic-net)

**CPCV:** 15 paths · AUC 0.698 ± 0.303 · **NO SIGNAL** (lower CI 0.50, marginal)

**Significant clusters:** none. All cluster MDA standard deviations exceed means.

> ho1s uses the **pooled `energy_all` model** (all 4 energy instruments combined);
> importance is scored on the **ho1s slice** of each CPCV test fold.
> The CPCV AUC variance (±0.303) is very high — the pooled model occasionally achieves
> strong ho1s discrimination in some fold combinations by chance, but not reliably.
> No cluster is significant; MDA–Coef agreement is poor (τ=0.31).
> Results are included for completeness only — no structure should be inferred.
> The top cluster by raw MDA (F5_signal, 0.173) has std=0.268, larger than its mean.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('ho1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.05, vmax=0.20)
    .set_caption('ho1s — cluster summary (energy_all pool, K=15 corr + 4 hand-assigned = 19 groups)')
)
show(BASE / 'ho1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

No significant clusters. MDA–Coef agreement poor (τ=0.31).


In [ ]:
show(BASE / 'ho1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'ho1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'ho1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    if 'coef_sum' in cc.columns: fmt['coef_sum'] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.20)
        .set_caption('ho1s — cluster cross-check (MDA · Coef; elastic-net logistic, energy_all pool)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement (MDA vs Coef):')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by raw MDA)

Members ranked by mean |coef|. PCA on train submatrix. No clusters are significant —
breakdowns shown for completeness only.


In [ ]:
wc = load(BASE / 'ho1s' / 'within_cluster_F5_signal.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'ho1s | F5_signal — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'ho1s' / 'within_cluster_F5_signal.png', width=950)


In [ ]:
wc = load(BASE / 'ho1s' / 'within_cluster_C13_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'ho1s | C13_f11 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'ho1s' / 'within_cluster_C13_f11.png', width=950)


In [ ]:
wc = load(BASE / 'ho1s' / 'within_cluster_C6_f2.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'ho1s | C6_f2 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'ho1s' / 'within_cluster_C6_f2.png', width=950)


### Step 5 · Global per-feature coefficient

Mean |standardised coefficient| per feature. Shown for structural reference;
no predictive interpretation is warranted given the lack of confirmed signal.


In [ ]:
gc = load(BASE / 'ho1s' / 'global_coef_summary.csv')
if not gc.empty:
    display(
        gc.head(25).style
        .format({'coef_abs': '{:.4f}', 'coef_signed': '{:.4f}'})
        .background_gradient(subset=['coef_abs'], cmap='Blues')
        .set_caption('ho1s — top 25 features by mean |coef| (NO SIGNAL — illustrative only)')
    )
show(BASE / 'ho1s' / 'global_coef_chart.png', width=1000)


---
## RB1S — champion: `energy_all` / Logistic (elastic-net)

**CPCV:** 15 paths · AUC 0.582 ± 0.094 · **NO SIGNAL** (lower CI 0.46)

**Significant clusters:** F5\_signal (MDA 0.092 ± 0.088) and C13\_f11 (MDA 0.073 ± 0.069) — both barely significant (mean ≈ std).

> rb1s shares the `energy_all` pool with ho1s/ng1s. Two clusters are technically significant
> (mean > std) but the instrument has no confirmed OOS signal (lower CI = 0.46). The significance
> likely reflects the pooled model finding some rb1s-relevant structure in the training data
> that doesn't survive OOS. MDA–Coef agreement is very poor (τ=0.17) — the elastic-net
> coefficients rank features differently than the permutation test, suggesting regularisation
> is suppressing features that matter for rb1s specifically.
> C13_f11 contains `f11_move_z` (coefficient 0.394 >> rest) but C13's PC1 explains only
> 27% of variance — a genuinely multi-dimensional cluster.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('rb1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.05, vmax=0.12)
    .set_caption('rb1s — cluster summary (energy_all pool, K=15 corr + 4 hand-assigned = 19 groups)')
)
show(BASE / 'rb1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

Two clusters reach significance (mean > std). MDA–Coef agreement poor (τ=0.17).


In [ ]:
show(BASE / 'rb1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'rb1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'rb1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    if 'coef_sum' in cc.columns: fmt['coef_sum'] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.12)
        .set_caption('rb1s — cluster cross-check (MDA · Coef; elastic-net logistic, energy_all pool)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement (MDA vs Coef):')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (significant clusters first, then top by MDA)

Members ranked by mean |coef|. PCA on train submatrix.


In [ ]:
wc = load(BASE / 'rb1s' / 'within_cluster_F5_signal.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'rb1s | F5_signal — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'rb1s' / 'within_cluster_F5_signal.png', width=950)


In [ ]:
wc = load(BASE / 'rb1s' / 'within_cluster_C13_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'rb1s | C13_f11 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'rb1s' / 'within_cluster_C13_f11.png', width=950)


In [ ]:
wc = load(BASE / 'rb1s' / 'within_cluster_C4_f2.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'rb1s | C4_f2 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'rb1s' / 'within_cluster_C4_f2.png', width=950)


### Step 5 · Global per-feature coefficient

Mean |standardised coefficient| per feature. No confirmed signal — shown for contrast.


In [ ]:
gc = load(BASE / 'rb1s' / 'global_coef_summary.csv')
if not gc.empty:
    display(
        gc.head(25).style
        .format({'coef_abs': '{:.4f}', 'coef_signed': '{:.4f}'})
        .background_gradient(subset=['coef_abs'], cmap='Blues')
        .set_caption('rb1s — top 25 features by mean |coef| (NO SIGNAL — illustrative only)')
    )
show(BASE / 'rb1s' / 'global_coef_chart.png', width=1000)


---
## NG1S — champion: `energy_all` / RF

**CPCV:** 12 paths (3 skipped) · AUC 0.409 ± 0.259 · **NO SIGNAL** (lower CI 0.22)

**Significant clusters:** none.

> ng1s has ~28 train events at the global cut (too thin for individual modelling, hence carried
> by `energy_all`). 3 of 15 CPCV folds were skipped because the ng1s test slice had fewer than
> 2 class labels. Mean AUC = 0.409 — below random, consistent with no signal. MDI–SHAP agree
> strongly (τ=0.87), which simply reflects the RF's consistent internal feature ranking —
> not predictive usefulness. MDA–MDI (τ=−0.17) and MDA–SHAP (τ=−0.18) actually disagree,
> indicating permutation-based importance is independent noise.
> Results are shown for diagnostic completeness only.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('ng1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.10, vmax=0.10)
    .set_caption('ng1s — cluster summary (energy_all pool, K=15 corr + 4 hand-assigned = 19 groups)')
)
show(BASE / 'ng1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

No significant clusters. MDA–MDI and MDA–SHAP disagree (negative τ).


In [ ]:
show(BASE / 'ng1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'ng1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'ng1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    for c in ['mdi_sum', 'shap_sum']: 
        if c in cc.columns: fmt[c] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.10, vmax=0.10)
        .set_caption('ng1s — cluster cross-check (MDA · MDI · SHAP ranks; RF, energy_all pool)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by raw MDA)

Members ranked by mean |SHAP|. No significant clusters — shown for diagnostic reference only.


In [ ]:
wc = load(BASE / 'ng1s' / 'within_cluster_C13_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'ng1s | C13_f11 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'ng1s' / 'within_cluster_C13_f11.png', width=950)


In [ ]:
wc = load(BASE / 'ng1s' / 'within_cluster_C9_f11_lowfreq_macro.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'ng1s | C9_f11_lowfreq_macro — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'ng1s' / 'within_cluster_C9_f11_lowfreq_macro.png', width=950)


In [ ]:
wc = load(BASE / 'ng1s' / 'within_cluster_C6_f2.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    import math
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'ng1s | C6_f2 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'ng1s' / 'within_cluster_C6_f2.png', width=950)


### Step 5 · Global per-feature SHAP

Mean |SHAP| per feature. Diagnostic only — below-random mean AUC (0.409).


In [ ]:
gs = load(BASE / 'ng1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('ng1s — top 25 features by mean|SHAP| (NO SIGNAL, below-random AUC)')
    )
show(BASE / 'ng1s' / 'global_shap_chart.png', width=1000)


---
## Cross-instrument findings — energy

Structured summary across cl1s, ho1s, rb1s, ng1s.


In [ ]:
ENERGY_INSTS = ['cl1s', 'ho1s', 'rb1s', 'ng1s']
NOSIGNAL = {'ho1s', 'rb1s', 'ng1s'}

rows = []
for inst in ENERGY_INSTS:
    mda  = load(BASE / inst / 'clustered_mda_full.csv')
    ra   = load(BASE / inst / 'rank_agreement.csv')
    meta = load(BASE / inst / 'champion_meta.csv')
    if mda.empty:
        continue
    top = mda.iloc[0]
    sig_count = int(mda['significant'].sum())
    tau_1 = ra['kendall_tau'].iloc[0] if not ra.empty else float('nan')
    champion = meta['model_type'].iloc[0].upper() if not meta.empty else '?'
    signal = meta['signal'].iloc[0] if not meta.empty else False
    rows.append({
        'inst': inst,
        'champion': champion,
        'signal': '\u2713' if signal else '\u2717',
        'n_sig_clusters': sig_count,
        'top_cluster': top['cluster'],
        'top_mda': round(top['mean_drop'], 4),
        'tau_primary': round(tau_1, 2) if not pd.isna(tau_1) else 'n/a',
    })

summary = pd.DataFrame(rows).set_index('inst')
display(summary.style.set_caption('Energy champions — cross-instrument summary'))


### Key observations

**CL1S** is the only confirmed signal in energy. It is driven overwhelmingly by a single
cluster of volatility/macro features (C14_f11, MDA 0.224) — roughly 50× the next cluster.
`f11_move_z` and `f2_vol_60` jointly define a volatility-regime latent factor. Crucially,
the internal F5_signal cluster is negative, implying crude oil signal quality features
do not contribute — predictability is entirely externally sourced from cross-asset fear indicators.

**HO1S/RB1S/NG1S** share the `energy_all` pooled model. No robust importance structure
is identifiable: ho1s has enormous fold-to-fold variance (±0.303); rb1s shows two borderline
significant clusters but no OOS signal; ng1s AUC is below random (0.409). The pooled model
learns some internal consistency across energy instruments but cannot produce reliable
ho1s/rb1s/ng1s-specific predictions — consistent with the selection table's lower CI values.

**C13_f11** (containing `f11_move_z`) ranks in the top 3 by raw MDA for *all four* energy
instruments, suggesting it is a common but noisy macro signal for energy markets.
Only for CL1S is it reliably above noise.
